In [1]:
import json
import time
import os
import re
import pandas as pd
from datetime import datetime
from vllm import LLM, SamplingParams

ideas:
1) https://www.youtube.com/watch?app=desktop&v=9j-480mlXEk&start=0

In [2]:
# Force vLLM to use the Data partition for everything
os.environ['VLLM_CONFIG_ROOT'] = '/Data/anahi_reyes/vllm_cache/config'
os.environ['VLLM_CACHE_ROOT'] = '/Data/anahi_reyes/vllm_cache'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['HF_HOME'] = '/Data/anahi_reyes/huggingface_cache'

# Load the in-disk LLM to the environment
model_path = '/Data/anahi_reyes/models/Mistral-7B-Instruct-v0.3'
llm = LLM(model=model_path, dtype='half', gpu_memory_utilization=0.7)
sampling_params = SamplingParams(temperature=0, max_tokens=512)
# max_tokens is output tokens


INFO 03-13 06:54:52 [utils.py:261] non-default args: {'dtype': 'half', 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'model': '/Data/anahi_reyes/models/Mistral-7B-Instruct-v0.3'}
INFO 03-13 06:54:52 [model.py:541] Resolved architecture: MistralForCausalLM
WARNING 03-13 06:54:52 [model.py:1885] Casting torch.bfloat16 to torch.float16.
INFO 03-13 06:54:52 [model.py:1561] Using max model len 32768
INFO 03-13 06:54:53 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-13 06:54:53 [vllm.py:624] Asynchronous scheduling is enabled.


Multiple tokenizer files found in directory: /Data/anahi_reyes/models/Mistral-7B-Instruct-v0.3. Using tokenizer.model.v3.


(EngineCore_DP0 pid=937691) INFO 03-13 06:54:53 [core.py:96] Initializing a V1 LLM engine (v0.15.0) with config: model='/Data/anahi_reyes/models/Mistral-7B-Instruct-v0.3', speculative_config=None, tokenizer='/Data/anahi_reyes/models/Mistral-7B-Instruct-v0.3', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_end

(EngineCore_DP0 pid=937691) Process EngineCore_DP0:
(EngineCore_DP0 pid=937691) Traceback (most recent call last):
(EngineCore_DP0 pid=937691)   File "/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore_DP0 pid=937691)     self.run()
(EngineCore_DP0 pid=937691)   File "/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/multiprocessing/process.py", line 108, in run
(EngineCore_DP0 pid=937691)     self._target(*self._args, **self._kwargs)
(EngineCore_DP0 pid=937691)   File "/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 950, in run_engine_core
(EngineCore_DP0 pid=937691)     raise e
(EngineCore_DP0 pid=937691)   File "/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 937, in run_engine_core
(EngineCore_DP0 pid=937691)  

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [12]:
df = df = pd.read_json("/Data/anahi_reyes/EDCD_data/raw_edcd_database_simple.jsonl", lines=True)
df.head()

,finess,hospital_name,nom_etab_long,keywords_nom_etab,source_url,title,content,score,retrieved_at
0,10000024,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,BOURG-EN-BRESSE FLEYRIAT,https://presse.ramsaygds.fr/communique/226176/Fermeture-temporaire-Urgences-de-Clinique-Ramsay-Convert?cm=1,Fermeture temporaire des Urgences de la Clinique Ramsay Convert,"Cette décision a été prise en concertation avec l’Agence Régionale de Santé, le Samu Centre 15 et le Centre Hospitalier Fleyriat de Bourg en Bresse.\n\nAinsi le service des urgences de la Clinique sera fermé la nuit et ne pourra plus accueillir de patient de 17h00 à 8h00, du Vendredi 11 Juillet 2025 au Mercredi 16 juillet.\n\nLe service des urgences sera ouvert chaque jour de 8h00 à 17h00.\n\nLes urgences cardiologiques restent quant à elles toujours assurées par l’établissement.\n\nPendant cette période, Il est demandé aux patients de se présenter au centre hospitalier de Fleyriat. Celui-ci recevra le renfort des médecins urgentistes de la clinique Ramsay Convert.\n\n#### À propos de Ramsay Santé\n\nÀ propos de Ramsay Santé [...] La Clinique Aguiléra lance les travaux de son nouveau service d’urgences (\n Vendredi 25 mars 2022\n\n Anthony Rablet nommé Directeur de la Clinique Convert à Bourg-en-Bresse (01) (\n Lundi 17 mai 2021\n\n L’Hôpital privé Jacques Cartier propose une prise en charge urologique unique en Essonne avec l’ouverture d’une filière ""SOS Calculs urinaires"" et l’acquisition d’un laser chirurgical de pointe (\n\nVoir tous les communiqués\n\n#### Le groupe Ramsay Santé\n\n ###### Le Groupe \n ###### Vous êtes patient \n ###### Vous êtes médecin \n ###### Rejoignez-nous \n ###### Actualités \n ###### Espace presse \n ###### Mon compte Ramsay Services \n\nMentions légales\n\n Image 64: fb\n Image 65: tt\n Image 66: instagram\n Image 67: li\n Image 68: u tube\n\npowered by PR-Rooms",0.718536,2026-02-19 00:58:52
1,10000024,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,BOURG-EN-BRESSE FLEYRIAT,https://www.franceinfo.fr/sante/hopital/urgences-un-acces-limite-a-bourg-en-bresse_5277172.html,Urgences : un accès limité à Bourg-en-Bresse - Franceinfo,"Publié\n\n Temps de lecture : 1min\n\nArticle rédigé par France 3 - F. Magnetto, S. Adam, M. Dubois\n\nFrance Télévisions\n\nÀ Bourg-en-Bresse, pour faire face à l'engorgement des urgences, l’accès est désormais régulé. Les patients ne peuvent plus se rendre spontanément aux urgences après 20 heures.\n\nLes urgences sont engorgées à Bourg-en-Bresse (Ain), pour éviter ce problème de fréquentation importante, les patients ne peuvent plus se présenter d’eux-mêmes au centre hospitalier après 20 heures. ""Je pensais qu’on allait me prendre en charge tout de suite quoi, je n'allais pas aller encore ailleurs"", s’étonne une patiente. ""On évalue le degré de gravité, le degré d’urgence du motif de consultation et en fonction de ce tri, on oriente"", explique Karine, infirmière d’accueil. [...] ## Une régulation obligatoire\n\nDans le but d’alléger la charge de travail, ce fonctionnement a été mis en place. Le service hospitalier est content de cette fluidification de la fréquentation qui limite les heures supplémentaires et permet au personnel de pouvoir s’occuper en priorité de la médecine. ""Ce qu’on aimerait bien, c’est s’occuper du cœur de notre métier qui est la médecine d’urgence et ce protocole permet un peu de retrouver du sens aussi en réorientant les patients"", rapporte Sébastien Roux, responsable des urgences de l’hôpital de Bourg-en-Bresse (Ain). L’autre problème de ce département, ce sont les déserts médicaux et le nombre peu élevé de médecins généralistes. Ainsi, les personnes ayant besoin d’un diagnostic rapide se rendent aux urgences. [...] • 2 min\n Pierre Vaultier fait la reconnaissance de la piste de snowboardcross\n\n • 3 min\n JO 2026 : le duo Guillaume Cizeron-Laurence Fournier Beaudry décroche l'or en danse sur glace\n\n • 2 min\n JO 2026 : la performance hypnotique du duo Loparev

In [ ]:
def build_extraction_prompt(row):
    h_name = row['hospital_name']
    h_keywords = row['keywords_nom_etab']

    return f"""<s>[INST]
You are extracting structured data about emergency department disruptions.

TARGET HOSPITAL:
"{h_name}" (also referenced as "{h_keywords}")

TASK:
Determine whether THIS hospital's emergency department (Urgences) is experiencing a disruption.

IMPORTANT ROLE IDENTIFICATION:
Determine the role of the target hospital. There could be more than one hospital mentioned, if so, identify  

Cases:
1. If THIS hospital is closing or regulating its emergency department → relevant = true
2. If THIS hospital is OPEN and only receiving redirected patients from another hospital → relevant = false
3. If the disruption concerns another hospital → relevant = false

TEMPORAL REASONING:
Identify the publication date of the article if present.
Use it to interpret relative expressions such as:
"today", "this weekend", "starting Monday".

OUTPUT FORMAT:
Return ONLY a valid JSON object. No explanations, no text outside JSON.

Possible outputs:

Relevant case:
{{
  "relevant": true,
  "status": "Full Closure | Temporary Closure | Access Regulated",
  "start_date": "YYYY-MM-DD | Unknown",
  "end_date": "YYYY-MM-DD | Ongoing | Unknown",
  "reason": "Staffing | Strike | Technical | Security | Other | Unknown",
  "publication_date": "YYYY-MM-DD | Unknown"
}}

Non-relevant case:
{{"relevant": false}}

TEXT:
{row['source_url']}
{row['title']}
{row['content']}
[/INST]"""

def robust_json_parse(raw_output):
    # This regex handles cases where the model adds text before or after the JSON
    match = re.search(r'(\{.*\})', raw_output, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            return {"relevant": False, "parsing_error": True}
    return {"relevant": False, "no_json_found": True}




# 3. Execution
# Note: Ensure your LLM generating function is defined (e.g., vLLM or LangChain)
prompts = df.apply(build_extraction_prompt, axis=1).tolist()
responses = llm.generate(prompts, sampling_params) # Adjust to your specific LLM caller


# Apply this to your vLLM outputs
df_ext = pd.DataFrame([robust_json_parse(r.outputs[0].text) for r in responses])

Adding requests:   0%|          | 0/20 [00:00<?, ?it/s]

/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/site-packages/mistral_common/tokens/tokenizers/sentencepiece.py:125: FutureWarning: `get_control_token` is deprecated. Use `get_special_token` instead.
  warnings.warn("`get_control_token` is deprecated. Use `get_special_token` instead.", FutureWarning)


Processed prompts:   0%|          | 0/20 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [20]:
df_ext.head()

,relevant,status,start_date,end_date,reason,publication_date
0,True,Temporary Closure,2025-07-11,2025-07-16,Staffing,2022-03-25
1,True,Access Regulated,Unknown,Ongoing,Overcrowding,Unknown
2,True,Access Regulated,2025-01-31,2025-11-07,Staffing,2025-10-16
3,False,NaN,NaN,NaN,NaN,NaN
4,False,NaN,NaN,NaN,NaN,NaN


In [21]:
df_final = pd.concat([df.reset_index(drop=True), df_ext.reset_index(drop=True)], axis=1)
df_final = pd.concat([df.reset_index(drop=True), df_ext.reset_index(drop=True)], axis=1)

In [22]:
pd.set_option('display.max_colwidth', None)
df_final.head()

,finess,hospital_name,nom_etab_long,keywords_nom_etab,source_url,title,content,score,retrieved_at,relevant,status,start_date,end_date,reason,publication_date
0,10000024,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,BOURG-EN-BRESSE FLEYRIAT,https://presse.ramsaygds.fr/communique/226176/Fermeture-temporaire-Urgences-de-Clinique-Ramsay-Convert?cm=1,Fermeture temporaire des Urgences de la Clinique Ramsay Convert,"Cette décision a été prise en concertation avec l’Agence Régionale de Santé, le Samu Centre 15 et le Centre Hospitalier Fleyriat de Bourg en Bresse.\n\nAinsi le service des urgences de la Clinique sera fermé la nuit et ne pourra plus accueillir de patient de 17h00 à 8h00, du Vendredi 11 Juillet 2025 au Mercredi 16 juillet.\n\nLe service des urgences sera ouvert chaque jour de 8h00 à 17h00.\n\nLes urgences cardiologiques restent quant à elles toujours assurées par l’établissement.\n\nPendant cette période, Il est demandé aux patients de se présenter au centre hospitalier de Fleyriat. Celui-ci recevra le renfort des médecins urgentistes de la clinique Ramsay Convert.\n\n#### À propos de Ramsay Santé\n\nÀ propos de Ramsay Santé [...] La Clinique Aguiléra lance les travaux de son nouveau service d’urgences (\n Vendredi 25 mars 2022\n\n Anthony Rablet nommé Directeur de la Clinique Convert à Bourg-en-Bresse (01) (\n Lundi 17 mai 2021\n\n L’Hôpital privé Jacques Cartier propose une prise en charge urologique unique en Essonne avec l’ouverture d’une filière ""SOS Calculs urinaires"" et l’acquisition d’un laser chirurgical de pointe (\n\nVoir tous les communiqués\n\n#### Le groupe Ramsay Santé\n\n ###### Le Groupe \n ###### Vous êtes patient \n ###### Vous êtes médecin \n ###### Rejoignez-nous \n ###### Actualités \n ###### Espace presse \n ###### Mon compte Ramsay Services \n\nMentions légales\n\n Image 64: fb\n Image 65: tt\n Image 66: instagram\n Image 67: li\n Image 68: u tube\n\npowered by PR-Rooms",0.718536,2026-02-19 00:58:52,True,Temporary Closure,2025-07-11,2025-07-16,Staffing,2022-03-25
1,10000024,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,BOURG-EN-BRESSE FLEYRIAT,https://www.franceinfo.fr/sante/hopital/urgences-un-acces-limite-a-bourg-en-bresse_5277172.html,Urgences : un accès limité à Bourg-en-Bresse - Franceinfo,"Publié\n\n Temps de lecture : 1min\n\nArticle rédigé par France 3 - F. Magnetto, S. Adam, M. Dubois\n\nFrance Télévisions\n\nÀ Bourg-en-Bresse, pour faire face à l'engorgement des urgences, l’accès est désormais régulé. Les patients ne peuvent plus se rendre spontanément aux urgences après 20 heures.\n\nLes urgences sont engorgées à Bourg-en-Bresse (Ain), pour éviter ce problème de fréquentation importante, les patients ne peuvent plus se présenter d’eux-mêmes au centre hospitalier après 20 heures. ""Je pensais qu’on allait me prendre en charge tout de suite quoi, je n'allais pas aller encore ailleurs"", s’étonne une patiente. ""On évalue le degré de gravité, le degré d’urgence du motif de consultation et en fonction de ce tri, on oriente"", explique Karine, infirmière d’accueil. [...] ## Une régulation obligatoire\n\nDans le but d’alléger la charge de travail, ce fonctionnement a été mis en place. Le service hospitalier est content de cette fluidification de la fréquentation qui limite les heures supplémentaires et permet au personnel de pouvoir s’occuper en priorité de la médecine. ""Ce qu’on aimerait bien, c’est s’occuper du cœur de notre métier qui est la médecine d’urgence et ce protocole permet un peu de retrouver du sens aussi en réorientant les patients"", rapporte Sébastien Roux, responsable des urgences de l’hôpital de Bourg-en-Bresse (Ain). L’autre problème de ce département, ce sont les déserts médicaux et le nombre peu élevé de médecins généralistes. Ainsi, les personnes ayant besoin d’un diagnostic rapide se rendent aux urgences. [...] • 2 min\n Pierre Vaultier fait la reconnaissance de la piste de snowboardcross\n\n • 3 min\n JO 2026 : le duo Guillaume Cizero

In [ ]:
def build_extraction_prompt(row):
    h_name = row['hospital_name']
    h_keywords = row['keywords_nom_etab']
    
    return f"""<s>[INST] You are a specialized data analyst. 

TASK:
Extract Emergency Room (Urgences) disruption data for: "{h_name}" (also known as "{h_keywords}"). Disruptions could be definitive (hospital closed completely), temporal (for a specified amount of time), or regulated access (closed at very specfic times of the day).

CHRONOLOGY RULES:
1. Identify the PUBLICATION DATE of the text first. This is your primary anchor.
2. Resolve relative terms (e.g., "yesterday", "last Friday", "next month") based on that Publication Date.

OUTPUT RULES:
- If no closure/regulation is mentioned for THIS hospital, return: {{"relevant": false}}
- Otherwise, return ONLY this JSON:
{{
  "relevant": true,
  "status": "Full Closure, Temporary Closure, or Access Regulated",
  "start_date": "YYYY-MM-DD (Calculated from publication date)",
  "end_date": "YYYY-MM-DD, Ongoing, or Unknown",
  "reason": "Staffing, Strike, Technical, or Security",
  "publication_date": "YYYY-MM-DD (The date the article/notice was issued)"
}}

CONTEXT:
{row['content']} [/INST]"""

# 3. Execution
# Note: Ensure your LLM generating function is defined (e.g., vLLM or LangChain)
prompts = df.apply(build_extraction_prompt, axis=1).tolist()
responses = llm.generate(prompts, sampling_params) # Adjust to your specific LLM caller

Adding requests:   0%|          | 0/20 [00:00<?, ?it/s]

/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/site-packages/mistral_common/tokens/tokenizers/sentencepiece.py:125: FutureWarning: `get_control_token` is deprecated. Use `get_special_token` instead.
  warnings.warn("`get_control_token` is deprecated. Use `get_special_token` instead.", FutureWarning)


Processed prompts:   0%|          | 0/20 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [7]:
responses[0]
responses[1]
responses[:5]

[RequestOutput(request_id=0, prompt='<s>[INST] You are a specialized health data analyst. \n\nTASK:\nExtract Emergency Room (Urgences) disruption data for: "CH DE FLEYRIAT" (also known as "BOURG-EN-BRESSE FLEYRIAT").\n\nCHRONOLOGY RULES:\n1. Identify the PUBLICATION DATE of the text first. This is your primary anchor.\n2. Resolve relative terms (e.g., "yesterday", "last Friday", "next month") based on that Publication Date.\n\nOUTPUT RULES:\n- If no closure/regulation is mentioned for THIS hospital, return: {"relevant": false}\n- Otherwise, return ONLY this JSON:\n{\n  "relevant": true,\n  "status": "Full Closure, Temporary Closure, or Access Regulated",\n  "start_date": "YYYY-MM-DD (Calculated from publication date)",\n  "end_date": "YYYY-MM-DD, Ongoing, or Unknown",\n  "reason": "Staffing, Strike, Technical, or Security",\n  "publication_date": "YYYY-MM-DD (The date the article/notice was issued)"\n}\n\nCONTEXT:\nCette décision a été prise en concertation avec l’Agence Régionale de S

In [ ]:
def build_extraction_prompt(row):
    # Use the retrieval date as the only temporal anchor
    retrieval_date = row['retrieved_at']
    
    return f"""<s>[INST] You are a specialized health data analyst. 
Context retrieval date: {retrieval_date}. Use this date to resolve all relative time expressions.

TASK:
Analyze the text to identify Emergency Room (Urgences) disruptions for {row['hospital_name']}.

RULES:
1. If the text does NOT contain specific information about a closure or regulation, return: {{"relevant": false}}
2. If it DOES contain relevant info, return this JSON:
{{
  "relevant": true,
  "status": "Full Closure, Temporary Closure, or Access Regulated",
  "start_date": "YYYY-MM-DD or Unknown",
  "end_date": "YYYY-MM-DD, Ongoing, or Unknown",
  "reason": "short categorized reason",
  "publication_date": "YYYY-MM-DD or Unknown"
}}

CONTEXT:
{row['content']} [/INST]"""

# 3. Execution
# Note: Ensure your LLM generating function is defined (e.g., vLLM or LangChain)
prompts = df.apply(build_extraction_prompt, axis=1).tolist()
responses = llm.generate(prompts, sampling_params) # Adjust to your specific LLM caller

extracted_data = []
for r in responses:
    try:
        # Standard cleaning of LLM output
        clean_json = r.outputs[0].text.strip().replace("```json", "").replace("```", "")
        extracted_data.append(json.loads(clean_json))
    except:
        extracted_data.append({"relevant": False, "error": "Parsing failed"})

# 4. Merging back to keep ALL original columns
df_ext = pd.DataFrame(extracted_data)
df_final = pd.concat([df.reset_index(drop=True), df_ext], axis=1)

In [ ]:
<s>[INST] You are a specialized data extractor for the French Ministry of Health.
Analyze the following text from a single search result for: {hospital_name}.

TASK:
1. Determine if the text mentions a specific Emergency Room disruption (Full Closure, Temporary/Partial Closure, or Regulated Access/15-SAMU call requirement).
2. If NO disruption is mentioned, or if the text is just general info/web navigation noise, return: {"is_relevant": false}.
3. If a disruption IS mentioned, extract the details below.

Today's date is {today}. Use it to calculate relative dates (e.g., "yesterday", "this coming Friday").

JSON Schema:
{
    "is_relevant": boolean,
    "closure_type": "Full Closure", "Temporary Closure", or "Access Regulated",
    "start_date": "YYYY-MM-DD",
    "end_date": "YYYY-MM-DD" or "Ongoing",
    "reason": "Categorized: Staff shortage, Technical, Strike, Renovation, or Other",
    "pub_date": "YYYY-MM-DD" (The date the news was published)
}

Return ONLY valid JSON.

CONTEXT:
{content} [/INST]

In [ ]:
# Read-in data
hospitals_info = pd.read_csv(
    "../data/intermediate/Emergency Room Lookup.csv",
    sep=";",          # ← this is the key
    encoding="utf-8",
)
df = pd.read_json("/Data/anahi_reyes/EDCD_data/raw_edcd_database.jsonl", lines=True)
df = df.head(10)

,id,finess,hospital_name,official_data,unofficial_data,retrieved_at
0,010000024G1,010000024,CH DE FLEYRIAT,[{'url': 'https://www.franceinfo.fr/sante/hopi...,[{'url': 'https://presse.ramsaygds.fr/Handlers...,2026-02-15 22:45:29
1,010000032G1,010000032,CH BUGEY SUD,[{'url': 'https://www.ch-bugeysud.fr/actualite...,"[{'url': 'https://www.ch-bugeysud.fr/', 'title...",2026-02-15 22:45:30
2,010005239G1,010005239,CH DU HAUT BUGEY - GEOVREISSET,[{'url': 'https://www.info-garde.com/ain/medec...,[{'url': 'https://www.ch-hautbugey.fr/category...,2026-02-15 22:45:31
3,010780195G1,010780195,CLINIQUE CONVERT,[{'url': 'https://www.facebook.com/leprogres.a...,[{'url': 'https://clinique-convert-bourg-en-br...,2026-02-15 22:45:32
4,010780203G1,010780203,HOPITAL PRIVE D AMBERIEU,[{'url': 'https://www.elsan.care/sites/default...,[{'url': 'https://www.samu-urgences-de-france....,2026-02-15 22:45:33


In [ ]:
df.shape

(715, 6)

In [ ]:
df.columns

Index(['id', 'finess', 'hospital_name', 'official_data', 'unofficial_data',
       'retrieved_at'],
      dtype='object')

In [ ]:
df.dtypes

id                         object
finess                     object
hospital_name              object
official_data              object
unofficial_data            object
retrieved_at       datetime64[ns]
dtype: object

In [ ]:
df.isna().sum().sort_values(ascending=False).head(20)

id                 0
finess             0
hospital_name      0
official_data      0
unofficial_data    0
retrieved_at       0
dtype: int64

In [ ]:
df.iloc[0].to_dict()

{'id': '010000024G1',
 'finess': '010000024',
 'hospital_name': 'CH DE FLEYRIAT',
 'official_data': [{'url': 'https://www.franceinfo.fr/sante/hopital/urgences-un-acces-limite-a-bourg-en-bresse_5277172.html',
   'title': 'Urgences : un accès limité à Bourg-en-Bresse - Franceinfo',
   'content': '## Une régulation obligatoire\n\nDans le but d’alléger la charge de travail, ce fonctionnement a été mis en place. Le service hospitalier est content de cette fluidification de la fréquentation qui limite les heures supplémentaires et permet au personnel de pouvoir s’occuper en priorité de la médecine. "Ce qu’on aimerait bien, c’est s’occuper du cœur de notre métier qui est la médecine d’urgence et ce protocole permet un peu de retrouver du sens aussi en réorientant les patients", rapporte Sébastien Roux, responsable des urgences de l’hôpital de Bourg-en-Bresse (Ain). L’autre problème de ce département, ce sont les déserts médicaux et le nombre peu élevé de médecins généralistes. Ainsi, les pers

In [8]:
# Official and unofficial data entries
df["n_official"] = df["official_data"].apply(lambda x: len(x) if isinstance(x, list) else 0)
df["n_unofficial"] = df["unofficial_data"].apply(lambda x: len(x) if isinstance(x, list) else 0)
df[["n_official", "n_unofficial"]].describe()
df.sort_values("n_official", ascending=False)[["finess","hospital_name","n_official","n_unofficial"]].head(10)

,finess,hospital_name,n_official,n_unofficial
0,010000024,CH DE FLEYRIAT,10,10
469,710978313,CENTRE HOSPITALIER JEAN BOUVERI,10,10
460,690807367,POLYCLINIQUE DU BEAUJOLAIS,10,10
461,700000011,GH HAUTE SAONE SITE GRAY,10,10
462,700000029,GH HAUTE SAONE SITE VESOUL,10,10
463,710010067,CH DU PAYS CHAROLAIS BRIONNAIS,10,10
465,710978263,CH WILLIAM MOREY CHALON SUR SAONE,10,10
466,710978263,CH WILLIAM MOREY CHALON SUR SAONE,10,10
467,710978289,CH LES CHANAUX MACON,10,10
468,710978289,CH LES CHANAUX MACON,10,10


In [9]:
def calculate_text_size(data_list):
    """Sums the characters in the 'content' field of a list of search results."""
    if not isinstance(data_list, list):
        return 0
    return sum(len(str(item.get('content', ''))) for item in data_list)

# 2. Apply the measurement to your DataFrame
df['official_chars'] = df['official_data'].apply(calculate_text_size)
df['unofficial_chars'] = df['unofficial_data'].apply(calculate_text_size)
df['total_chars'] = df['official_chars'] + df['unofficial_chars']

# 3. Quick Stats Report
report = {
    "Total Hospitals": len(df),
    "Avg Official Chars": df['official_chars'].mean(),
    "Avg Unofficial Chars": df['unofficial_chars'].mean(),
    "Max Total Chars": df['total_chars'].max(),
    "Hospitals with 0 Data": len(df[df['total_chars'] == 0])
}

print("--- DATA SIZE REPORT ---")
for key, value in report.items():
    print(f"{key}: {value:,.0f}")

--- DATA SIZE REPORT ---
Total Hospitals: 715
Avg Official Chars: 17,739
Avg Unofficial Chars: 18,677
Max Total Chars: 78,563
Hospitals with 0 Data: 0


In [10]:
# Show top 5 hospitals with the most data
print("\nHospitals with the most retrieved text:")
print(df.sort_values('total_chars', ascending=False)[['hospital_name', 'total_chars']].head())


Hospitals with the most retrieved text:
                          hospital_name  total_chars
44   CH ARIEGE COUSERANS SITE ST LIZIER        78563
503      HU PARIS NORD SITE BICHAT APHP        47077
630     CH DES DEUX VALLEES SITE JUVISY        46916
662     HU PARIS SITE JEAN VERDIER APHP        46577
661     HU PARIS SITE JEAN VERDIER APHP        46577


In [11]:
# Check how many hospitals have data beyond the 40k limit
hospitals_exceeding = df[df['total_chars'] > 40000]
print(f"Hospitals exceeding 40k limit: {len(hospitals_exceeding)} ({len(hospitals_exceeding)/len(df)*100:.1f}%)")

# Average loss for those exceeding
avg_loss = (hospitals_exceeding['total_chars'] - 40000).mean()
print(f"Average characters truncated for those over limit: {avg_loss:,.0f}")

Hospitals exceeding 40k limit: 244 (34.1%)
Average characters truncated for those over limit: 2,633


We need to analyze all the text, but the size of the tests are different for each type of retrieval: official vs unofficial. We set different parameters for the two types of retrievals that we need to perform

In [ ]:
# --- 1. Preparación del contexto con índices de fuente ---
def prepare_context_v3(row):
    all_snippets = []
    # Combinamos fuentes oficiales y no oficiales
    sources = row.get('unofficial_data', []) + row.get('official_data', [])
    
    for i, item in enumerate(sources):
        url = item.get('url', 'Unknown URL')
        content = item.get('content', '')
        if content:
            # Marcamos cada fuente claramente para que el LLM pueda citarla
            header = f"\n[SOURCE_{i} URL: {url}]\n"
            all_snippets.append(header + content.replace('\n', ' ').strip())
    
    return "\n".join(all_snippets)[:40000]

df['full_context'] = df.apply(prepare_context_v3, axis=1)

# --- 2. Definición del Prompt con Anclaje Temporal ---
def build_final_prompt(row):
    today = datetime.now().strftime("%Y-%m-%d")
    return f"""<s>[INST] You are a specialized data extractor for the French Ministry of Health (DREES).
Today's date is: {today}. Use it to resolve relative dates.

TASK: Extract all Emergency Room disruptions for: {row['hospital_name']}.
Include: Total closures, night closures, 15/SAMU regulations, or Cyberattacks.

FORMAT: Return ONLY a valid JSON list of objects.
Keys: 
- "start_date" (YYYY-MM-DD)
- "end_date" (YYYY-MM-DD or "Ongoing")
- "event_type" (Fermeture, Régulation, Cyberattaque, Grève)
- "reason" (Categorized reason in English: Staffing, Technical, External, or Strike)
- "source_url" (The EXACT URL provided in the [SOURCE_X] tag where you found the info)

CONTEXT:
{row['full_context']} [/INST]"""

# --- 3. Ejecución y Estructuración ---
print("Building prompts...")
prompts = df.apply(build_final_prompt, axis=1).tolist()

inference_params = SamplingParams(temperature=0.0, max_tokens=1000)
outputs = llm.generate(prompts, inference_params)

extracted_events = []
for i, output in enumerate(outputs):
    raw_text = output.outputs[0].text.strip()
    original_row = df.iloc[i]
    
    try:
        clean_json = raw_text.replace("```json", "").replace("```", "").strip()
        events = json.loads(clean_json)
        
        for ev in events:
            # Añadimos metadatos de trazabilidad
            ev['hospital_name'] = original_row['hospital_name']
            ev['finess'] = original_row['finess']
            ev['retrieved_at'] = original_row['retrieved_at']
            extracted_events.append(ev)
    except Exception as e:
        print(f"Error en {original_row['hospital_name']}: {e}")

df_final = pd.DataFrame(extracted_events)

Building prompts...


Adding requests:   0%|          | 0/715 [00:00<?, ?it/s]

/users/eleves-b/2024/anahi.reyes-miguel/miniconda3/envs/vllm_env/lib/python3.10/site-packages/mistral_common/tokens/tokenizers/sentencepiece.py:125: FutureWarning: `get_control_token` is deprecated. Use `get_special_token` instead.
  warnings.warn("`get_control_token` is deprecated. Use `get_special_token` instead.", FutureWarning)


Processed prompts:   0%|          | 0/715 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [ ]:
df_final.head()

: 

In [ ]:
# 1. Build the Prompts
# We use a strict instruction set to ensure Mistral 0.3 behaves as a data parser
def build_prompt_with_sources(row):
    today = datetime.now().strftime("%Y-%m-%d")
    return f"""<s>[INST] You are a specialized data extractor for the DREES.
    Today's date is: {today}.
    
    TASK: Extract Emergency Room disruptions (closures, regulations, cyberattacks) for: {row['hospital_name']}.
    
    For each event found, you MUST include the URL of the source used.
    
    FORMAT: Return ONLY a JSON list of objects with these keys:
    "start_date", "end_date", "event_type", "reason", "source_url"

    CONTEXT:
    {row['full_context']} [/INST]"""

print("Building prompts...")
prompts = df.apply(build_final_prompt, axis=1).tolist()

# 2. Launch vLLM Inference
# We use temperature 0.0 for deterministic, factual results
from vllm import SamplingParams
inference_params = SamplingParams(temperature=0.0, max_tokens=1000)

print(f"Processing {len(prompts)} hospitals on GPU...")
outputs = llm.generate(prompts, inference_params)

# 3. Flatten the JSON results into a Clean Table
extracted_events = []

for i, output in enumerate(outputs):
    raw_text = output.outputs[0].text.strip()
    original_row = df.iloc[i]
    
    try:
        # Clean potential Markdown wrapping
        clean_json = raw_text.replace("```json", "").replace("```", "").strip()
        events = json.loads(clean_json)
        
        for ev in events:
            # Map metadata back to each individual event
            ev['hospital_name'] = original_row['hospital_name']
            ev['finess'] = original_row['finess']
            ev['record_id'] = original_row['id']
            extracted_events.append(ev)
            
    except Exception as e:
        # If a specific hospital fails, we log it but keep the loop running
        print(f"Error parsing {original_row['hospital_name']}: {e}")

# 4. Final Dataframe Construction
df_final = pd.DataFrame(extracted_events)

# Reorder columns for the DREES report
final_cols = ['hospital_name', 'finess', 'start_date', 'end_date', 'event_type', 'reason']
df_final = df_final[final_cols]

# Save to CSV
df_final.to_csv("/Data/anahi_reyes/EDCD_data/emergency_closures_final_report.csv", index=False)
print(f"Success! Extracted {len(df_final)} events across 715 hospitals.")

In [25]:
def prepare_context(row):
    """Smarter concatenation: prioritizes news snippets and handles high volume."""
    # 1. Start with Unofficial Data (usually News = higher signal, lower noise)
    unofficial_snippets = [res.get('content', '') for res in row.get('unofficial_data', [])]
    
    # 2. Add Official Data (usually PDFs = high noise)
    official_snippets = [res.get('content', '') for res in row.get('official_data', [])]
    
    # Join them with a clear separator
    combined_text = "\n[NEWS SOURCES]\n" + "\n---\n".join(unofficial_snippets)
    combined_text += "\n[OFFICIAL DECREES]\n" + "\n---\n".join(official_snippets)
    
    # NEW LIMIT: 40,000 characters (~10,000 tokens)
    # This covers your AVERAGE case (36k) perfectly and fits comfortably in Mistral's memory
    return combined_text[:40000]

print("Preparando contextos para el LLM...")
df['full_context'] = df.apply(prepare_context, axis=1)


Preparando contextos para el LLM...


In [ ]:
# --- 1. PREPARACIÓN DE DATOS ---
def prepare_context(row):
    """Concatena official y unofficial data en un solo string limpio"""
    all_snippets = []
    # Unimos ambas listas de diccionarios
    sources = row.get('official_data', []) + row.get('unofficial_data', [])
    
    for item in sources:
        content = item.get('content', '')
        if content:
            # Limpieza básica de ruido visual
            clean_content = content.replace('\n', ' ').strip()
            all_snippets.append(clean_content)
    
    # Unimos todo con separadores claros. 
    # Limitamos a 8000 caracteres para no exceder el contexto del modelo 7B
    return " | ".join(all_snippets)[:8000]

print("Preparando contextos para el LLM...")
df['full_context'] = df.apply(prepare_context, axis=1)

# --- 2. CONFIGURACIÓN DEL PROMPT ---
def build_prompt(row):
    return f"""<s>[INST] You are a specialized data extractor for the French Ministry of Health.
Analyze the provided context about the hospital: {row['hospital_name']} (FINESS: {row['finess']}).

TASK: Extract all events related to Emergency Room (Urgences) between 2022 and 2026.
FIND: 
- Total closures (Fermeture totale)
- Night closures (Fermeture nocturne)
- Mandatory 15/SAMU regulation (Accès régulé)
- Cyberattacks impacting service.

FORMAT: Return ONLY a valid JSON list of objects. If no events are found, return [].
Keys: "start_date" (YYYY-MM-DD), "end_date" (YYYY-MM-DD or "Ongoing"), "event_type", "reason" (short English summary).

CONTEXT:
{row['full_context']} [/INST]"""

prompts = df.apply(build_prompt, axis=1).tolist()

# --- 3. INFERENCIA CON vLLM ---
# Usamos temperatura 0 para máxima precisión en datos
sampling_params = SamplingParams(temperature=0.0, max_tokens=600)

print(f"Lanzando inferencia masiva para {len(prompts)} hospitales...")
outputs = llm.generate(prompts, sampling_params)

# --- 4. ESTRUCTURACIÓN DEL DATAFRAME FINAL ---
final_events = []

for i, output in enumerate(outputs):
    generated_text = output.outputs[0].text.strip()
    original_row = df.iloc[i]
    
    try:
        # Extraer el JSON del texto (por si el modelo pone etiquetas ```json)
        if "```json" in generated_text:
            generated_text = generated_text.split("```json")[1].split("```")[0]
        elif "```" in generated_text:
            generated_text = generated_text.split("```")[1].split("```")[0]
            
        events = json.loads(generated_text)
        
        for ev in events:
            # Añadimos los metadatos del hospital a cada evento encontrado
            ev['hospital_id'] = original_row['id']
            ev['finess'] = original_row['finess']
            ev['hospital_name'] = original_row['hospital_name']
            final_events.append(ev)
            
    except Exception as e:
        print(f"Error parseando resultados de {original_row['hospital_name']}: {e}")

# Convertir a DataFrame plano
df_final = pd.DataFrame(final_events)

# --- 5. EXPORTACIÓN ---
output_file = "/Data/anahi_reyes/EDCD_data/emergency_closures_final.csv"
df_final.to_csv(output_file, index=False)

print(f"\n✅ ¡Extracción completada! Se encontraron {len(df_final)} eventos.")
print(f"Archivo guardado en: {output_file}")

# Ver los primeros resultados
df_final.head()